In [16]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# === Load from your Downloads (same as your original) ===
df = pd.read_csv("/Users/kimballweeks/Downloads/cleaned_data.csv")

# === Same renames (adds edu 2023) ===
df = df.rename(columns={
    "pct_highschool_or_more (1990)": "pct_hs_1990",
    "pct_highschool_or_more_(2023)": "pct_hs_2023",   # <-- NEW
    "Pop 2023": "pop_2023",
    "Established firms 1989": "firms_1989",
    "Established firms 2022": "firms_2022",
    "income_1989_real_2023": "income_1989_real"
    # keep "church_persistence_index" as-is
})

# === Ensure numeric ===
for col in ["church_persistence_index","income_1989_real","pct_hs_1990","pct_hs_2023",
            "pop_2023","firms_1989","firms_2022"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# --- Guard against log(0) / negatives (prevents SVD) ---
for v in ["firms_2022","pop_2023","firms_1989"]:
    df.loc[df[v] <= 0, v] = np.nan

# === Logs + standardization (as in your baseline) ===
df["log_firms_2022"] = np.log(df["firms_2022"])
df["log_pop_2023"]   = np.log(df["pop_2023"])
df["log_firms_1989"] = np.log(df["firms_1989"])

df["income_1989_std"] = (df["income_1989_real"] - df["income_1989_real"].mean()) / df["income_1989_real"].std()
df["pct_hs_1990_std"] = (df["pct_hs_1990"]      - df["pct_hs_1990"].mean())      / df["pct_hs_1990"].std()

# === Build ONE common sample used for BOTH regressions ===
vars_all = ["log_firms_2022", "church_persistence_index", "income_1989_std",
            "pct_hs_1990_std", "log_pop_2023", "log_firms_1989",
            "pct_hs_2023", "State"]
d = df[vars_all].replace([np.inf,-np.inf], np.nan).dropna()

# Keep states with >=2 obs so C(State) is estimable
vc = d["State"].value_counts()
d = d[d["State"].isin(vc[vc > 1].index)]

print(f"Common N = {len(d)}, States = {d['State'].nunique()}")

# === BASELINE (same spec) on the common sample ===
baseline = smf.ols(
    "log_firms_2022 ~ church_persistence_index + income_1989_std + pct_hs_1990_std + "
    "log_pop_2023 + log_firms_1989 + C(State)",
    data=d
).fit(cov_type="HC3")

print("\n--- Baseline ---")
print(baseline.summary())

# === MEDIATION: add Education 2023 (the ONLY modeling change) ===
mediation = smf.ols(
    "log_firms_2022 ~ church_persistence_index + income_1989_std + pct_hs_1990_std + "
    "log_pop_2023 + log_firms_1989 + pct_hs_2023 + C(State)",
    data=d
).fit(cov_type="HC3")

print("\n--- Mediation (Education 2023 added) ---")
print(mediation.summary())

# Attenuation (now apples-to-apples because sample is identical)
b0 = baseline.params.get("church_persistence_index", np.nan)
b1 = mediation.params.get("church_persistence_index", np.nan)
if np.isfinite(b0) and np.isfinite(b1) and b0 != 0:
    print(f"\nPersistence coef: {b0:.4f}  →  {b1:.4f} (with edu_2023); "
          f"attenuation = {(100*(b0-b1)/b0):.1f}%")


Common N = 2968, States = 48

--- Baseline ---
                            OLS Regression Results                            
Dep. Variable:         log_firms_2022   R-squared:                       0.963
Model:                            OLS   Adj. R-squared:                  0.963
Method:                 Least Squares   F-statistic:                     1605.
Date:                Wed, 13 Aug 2025   Prob (F-statistic):               0.00
Time:                        13:57:06   Log-Likelihood:                -567.54
No. Observations:                2968   AIC:                             1241.
Df Residuals:                    2915   BIC:                             1559.
Df Model:                          52                                         
Covariance Type:                  HC3                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------